---

# DTL Model Pipeline DTL-v08

Notebook mandiri untuk ID3, C4.5, dan CART dari nol. Pipeline akhir adalah **CART DTL-v08** dengan pemangkasan pascalatih dan ambang klasifikasi yang dipilih melalui validasi silang. Jalankan semua sel berurutan dari akar repositori atau folder `notebooks/`. Notebook hanya membutuhkan `data/` dan pustaka yang tercantum di bawah, tanpa folder `src/`.

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Model Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)

---

# Introduction <a name="1"></a>

---

Eksperimen ini memprediksi pembatalan pemesanan (`is_canceled`) dari data pemesanan hotel. Ketiga varian pohon dibandingkan pada satu holdout yang sama. Pemilihan konfigurasi pipeline final hanya menggunakan bagian latih melalui validasi silang. Holdout dipakai sekali untuk melaporkan hasil akhir.

**DTL-v08** adalah CART dari nol dengan kriteria Gini, konfigurasi pertumbuhan, *cost-complexity post-pruning*, dan ambang probabilitas yang dipilih dengan *5-fold stratified cross-validation*. ID3 dan C4.5 tetap dilatih sebagai baseline, bukan kandidat yang dipilih ulang berdasarkan skor holdout. Skor historis 0,8371 berasal dari hasil eksperimen tersimpan. Skor notebook ini dihitung ulang ketika dijalankan.

## Metric

Metrik utama adalah *Macro F1* pada validation dengan kedua kelas diberi bobot sama.

$$\mathrm{MacroF1}=\frac{1}{2}\sum_{c\in\{0,1\}}\mathrm{F1}_c$$

Gunakan `labels=[0, 1]`, `average='macro'`, dan `zero_division=0`.

## Dataset

Input berasal dari `data/train.csv` dan `data/test.csv`. `is_canceled` adalah target, sedangkan `id` dipertahankan untuk submission dan dikeluarkan dari fitur.

Split stratifikasi seed 42 dibuat sebelum eksplorasi berlabel. Setiap fold validasi silang melatih ulang imputasi dan pohon pada bagian latih fold tersebut. Seluruh kode algoritma disertakan dalam notebook agar artefak ini dapat dijalankan tanpa `src/`.

---

# Initialization <a name="2"></a>

---

Impor pustaka, temukan folder `data/`, dan buat holdout bersama sebelum menelaah target.

## Environment Setup

Uncomment installation only when the environment is missing these libraries.

In [1]:
# %pip install -q numpy pandas scikit-learn

Hanya NumPy, pandas, dan scikit-learn yang diperlukan. Penyimpanan model menggunakan `pickle` dari pustaka standar. Jangan memuat artefak dari sumber yang tidak dipercaya.

Modul berikut digunakan untuk pohon dari nol, evaluasi, validasi silang, pembanding, dan serialisasi.

In [2]:
import pickle
from copy import copy, deepcopy
from dataclasses import dataclass
from html import escape
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

## Settings

In [3]:
class Settings:
    SEED = 42
    MODEL_NAME = 'dtl'
    ROOT_DIR = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'data' / 'train.csv').is_file()), Path.cwd())
    DATA_DIR = ROOT_DIR / 'data'
    TRAIN_PATH = DATA_DIR / 'train.csv'
    TEST_PATH = DATA_DIR / 'test.csv'
    OUTPUT_DIR = ROOT_DIR / 'output'
    MODEL_DIR = OUTPUT_DIR / 'final' / 'model' / MODEL_NAME
    TREE_TOP_N = 10  # Number of split nodes to show, breadth-first from the root.

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Dataset

In [4]:
train_df = pd.read_csv(CFG.TRAIN_PATH, dtype={'id': str})
test_df = pd.read_csv(CFG.TEST_PATH, dtype={'id': str})
print(f'Train: {train_df.shape} | Test: {test_df.shape}')

# Create the holdout before any target-aware exploration.
y_all = train_df['is_canceled'].to_numpy()
if not np.isin(y_all, [0, 1]).all():
    raise ValueError('Target is expected to contain only labels 0 and 1.')
rng = np.random.default_rng(CFG.SEED)
valid_idx = np.sort(np.concatenate([
    rng.permutation(np.flatnonzero(y_all == label))[:int(np.ceil(0.2 * np.sum(y_all == label)))]
    for label in (0, 1)
]))
train_idx = np.setdiff1d(np.arange(len(train_df)), valid_idx)
train_set = train_df.iloc[train_idx].copy()
val_set = train_df.iloc[valid_idx].copy()

Train: (27373, 19) | Test: (6844, 18)


Holdout mencakup sekitar 20% dari setiap kelas, dipilih dengan generator NumPy seed 42. Indeks ini tidak diubah berdasarkan hasil model.

---

# Data Understanding <a name="3"></a>

---

Gunakan bagian ini untuk memahami data yang tersedia bagi model ini. Pemeriksaan yang memakai target dilakukan hanya pada `train_set`.

## Describe Data

In [5]:
feature_columns = [column for column in train_df.columns if column not in {'id', 'is_canceled'}]
train_features = train_set[feature_columns]
feature_profile = pd.DataFrame({
    'dtype': train_features.dtypes.astype(str),
    'missing': train_features.isna().sum(),
    'missing_pct': train_features.isna().mean().mul(100).round(2),
    'unique': train_features.nunique(dropna=True),
})
display(feature_profile)

target_counts = train_set['is_canceled'].value_counts().reindex([0, 1], fill_value=0)
display(pd.DataFrame({'count': target_counts, 'proportion': target_counts / len(train_set)}, index=[0, 1]))

,dtype,missing,missing_pct,unique
num_adults,float64,652,2.98,5
num_children,float64,2453,11.20,6
weekend_nights,int64,0,0.00,8
weekday_nights,int64,0,0.00,18
meal_package,object,1413,6.45,4
needs_parking,float64,865,3.95,2
room_category,object,811,3.70,7
days_before_arrival,float64,909,4.15,403
arrival_year,int64,0,0.00,2
arrival_month,int64,0,0.00,12


,count,proportion
0,14723,0.672345
1,7175,0.327655


Profil fitur dan proporsi kelas dihitung pada `train_set` saja. Ringkasan ini membantu mendeteksi nilai kosong dan tipe fitur sebelum menetapkan strategi imputasi.

## Explore Data

In [6]:
numeric_features = train_features.select_dtypes(include='number').columns
categorical_features = train_features.select_dtypes(exclude='number').columns

display(train_set.groupby('is_canceled')[numeric_features].median().T.rename(columns={0: 'not_canceled', 1: 'canceled'}))
for column in categorical_features:
    category_summary = train_set.assign(category=train_set[column].fillna('__MISSING__')).groupby('category', dropna=False)['is_canceled'].agg(count='size', cancellation_rate='mean')
    display(category_summary.sort_values('count', ascending=False))

is_canceled,not_canceled,canceled
num_adults,2.0,2.0
num_children,0.0,0.0
weekend_nights,1.0,1.0
weekday_nights,2.0,2.0
needs_parking,0.0,0.0
days_before_arrival,39.0,122.0
arrival_year,2024.0,2024.0
arrival_month,8.0,8.0
arrival_day,15.0,16.0
is_returning_guest,0.0,0.0


,count,cancellation_rate
category,,
Breakfast,15716,0.310957
Room Only,2981,0.338812
Half Board,1784,0.459081
__MISSING__,1413,0.324133
Full Board,4,0.250000


,count,cancellation_rate
category,,
Type B,16325,0.320674
Type G,3507,0.341888
__MISSING__,811,0.361282
Type A,572,0.430070
Type D,429,0.328671
Type C,152,0.236842
Type F,96,0.239583
Type E,6,0.333333


,count,cancellation_rate
category,,
Online Travel Agent,14036,0.363138
Offline Agent,6357,0.301715
Corporate,1189,0.116905
Complimentary,241,0.000000
Airline,75,0.280000


Ringkasan bersyarat target terbatas pada bagian latih. Perbedaan median atau tingkat pembatalan kategori bersifat deskriptif, bukan bukti pengaruh kausal.

## Verify Data Quality

In [7]:
test_features = test_df.drop(columns='id')
quality_checks = {
    'unique IDs in train': train_df['id'].is_unique,
    'unique IDs in test': test_df['id'].is_unique,
    'train and test feature columns match': set(feature_columns) == set(test_features.columns),
    'binary training target': set(train_set['is_canceled'].unique()).issubset({0, 1}),
}
display(pd.Series(quality_checks, name='passed').to_frame())
if not all(quality_checks.values()):
    raise ValueError('Review data quality checks before continuing.')

,passed
unique IDs in train,True
unique IDs in test,True
train and test feature columns match,True
binary training target,True


Pemeriksaan kesesuaian kolom dan keunikan ID mencegah kesalahan pemetaan prediksi ke baris submission. Ketidaksesuaian menghentikan notebook sebelum pelatihan.

---

# Training and Validation Split <a name="4"></a>

---

Split sudah dibuat di Initialization sebelum eksplorasi target-aware agar `train_set` dapat digunakan pada bagian Data Understanding.

In [8]:
split_class_counts = pd.DataFrame({
    'train': train_set['is_canceled'].value_counts().reindex([0, 1], fill_value=0),
    'validation': val_set['is_canceled'].value_counts().reindex([0, 1], fill_value=0),
})
display(split_class_counts)

cross_split_duplicates = train_set[feature_columns].drop_duplicates().merge(
    val_set[feature_columns].drop_duplicates(), on=feature_columns, how='inner'
).shape[0]
print(f'Identical feature rows across train and validation: {cross_split_duplicates}')

,train,validation
is_canceled,,
0,14723,3681
1,7175,1794


Identical feature rows across train and validation: 25


Tabel jumlah kelas memeriksa stratifikasi. Baris fitur identik pada kedua bagian hanya dilaporkan. Pemisahan dilakukan menurut baris, tanpa menghapus data otomatis.

---

# Data Preparation <a name="5"></a>

---

Persiapan fitur mengikuti semantik runner DTL dengan median untuk nilai numerik kosong dan penanda khusus untuk kategori kosong atau belum terlihat. Semua statistik dipelajari dari data latih yang berlaku.

## Select Data

In [9]:
X_train_raw = train_set[feature_columns].copy()
X_val_raw = val_set[feature_columns].copy()
X_test_raw = test_df[feature_columns].copy()
y_train = train_set['is_canceled'].to_numpy(dtype=int)
y_val = val_set['is_canceled'].to_numpy(dtype=int)

if X_train_raw.columns.tolist() != X_val_raw.columns.tolist() or X_train_raw.columns.tolist() != X_test_raw.columns.tolist():
    raise ValueError('Train, validation, and test feature columns must match in the same order.')
if {'id', 'is_canceled'} & set(feature_columns):
    raise ValueError('ID and target must not be used as model features.')
print(f'Features: {len(feature_columns)} | train: {len(X_train_raw)} | validation: {len(X_val_raw)} | test: {len(X_test_raw)}')

Features: 17 | train: 21898 | validation: 5475 | test: 6844


`id` hanya digunakan untuk pemetaan keluaran. Urutan baris dan nama kolom fitur dipertahankan pada latih, validasi, dan test.

## Clean Data

### Missing Values

In [10]:
missing_summary = pd.DataFrame({
    'train': X_train_raw.isna().sum(),
    'validation': X_val_raw.isna().sum(),
    'test': X_test_raw.isna().sum(),
})
display(missing_summary.loc[missing_summary.any(axis=1)].sort_values('train', ascending=False))

,train,validation,test
num_children,2453,582,747
special_requests,2029,471,593
nightly_rate_idr,1640,398,540
meal_package,1413,347,442
days_before_arrival,909,214,290
needs_parking,865,217,247
room_category,811,191,242
num_adults,652,157,180


Median numerik dihitung dari bagian latih saja. Kolom numerik yang seluruhnya kosong diisi `0.0`. Kategori kosong diberi `__MISSING__`, sedangkan kategori baru saat prediksi diberi `__UNKNOWN__`.

### Duplicates

In [11]:
duplicate_mask = X_train_raw.duplicated(subset=feature_columns, keep=False)
print(f'Training rows with duplicate features: {int(duplicate_mask.sum())}')

Training rows with duplicate features: 79


Duplikasi fitur tidak dihapus karena dua baris dengan fitur sama dapat memiliki target berbeda. Penghapusan sepihak akan mengubah distribusi pelatihan.

### Outliers

In [12]:
numeric_columns = X_train_raw.select_dtypes(include='number').columns
quartiles = X_train_raw[numeric_columns].quantile([0.25, 0.75])
iqr = quartiles.loc[0.75] - quartiles.loc[0.25]
lower, upper = quartiles.loc[0.25] - 1.5 * iqr, quartiles.loc[0.75] + 1.5 * iqr
outlier_counts = ((X_train_raw[numeric_columns] < lower) | (X_train_raw[numeric_columns] > upper)).sum()
display(outlier_counts.rename('iqr_outliers').to_frame().sort_values('iqr_outliers', ascending=False))

,iqr_outliers
num_adults,5850
arrival_year,3925
num_children,1626
nightly_rate_idr,924
days_before_arrival,739
needs_parking,637
is_returning_guest,546
prior_completed_bookings,476
special_requests,436
weekday_nights,200


IQR hanya dipakai untuk diagnosis. Pohon membagi berdasarkan urutan dan ambang nilai. Nilai ekstrem tidak dipangkas tanpa pengujian terpisah.

### Data Errors

Nilai di luar domain atau tipe data yang salah perlu diperiksa sebelum menjalankan notebook. Tidak ada koreksi otomatis di luar imputasi nilai kosong.

## Construct Data

### Attribute Construction

Tidak ada fitur turunan. Pohon menggunakan fitur yang tersedia agar hasil tetap sebanding dengan runner eksperimen.

### Feature Scaling

Penskalaan tidak diperlukan karena keputusan pada pohon didasarkan pada pemisahan nilai, bukan jarak antarsampel.

### Encoding

Pohon dari nol menerima kategori sebagai teks. Pembanding scikit-learn memakai *one-hot encoding* karena `DecisionTreeClassifier` membutuhkan input numerik.

### Dimensionality Reduction

Tidak diterapkan seleksi fitur atau reduksi dimensi. Semua fitur selain `id` dan target dipertahankan.

## Class Balance

Tidak ada pembobotan kelas atau pengambilan sampel ulang pada DTL-v08. Eksperimen *class weighting* DTL-v09 merupakan jalur terpisah dan tidak menentukan pipeline final.

## Format Data

Implementasi dari nol menerima `DataFrame` berisi angka terimputasi dan kategori teks. Pembanding menerima matriks numerik dengan kategori hasil *one-hot encoding*.

---

# Model Pipeline <a name="6"></a>

---

`SharedPreprocessor` dan `BaselineTree` berikut disalin dari implementasi eksperimen saat notebook disusun, bukan diimpor saat runtime. ID3 memilih *information gain*. C4.5 menyaring kandidat berdasarkan gain lalu memilih *gain ratio*. CART memilih penurunan Gini terbesar. Kategori ID3/C4.5 dipisah multi-cabang, sedangkan CART memakai partisi biner. Calon ambang numerik dibatasi 64 dan kategori dengan lebih dari 10 nilai memakai calon *one-versus-rest*.

Pemangkasan pascalatih membandingkan galat daun berbobot dan galat subpohon ditambah penalti `alpha` per daun. Karena penalti menaikkan biaya pohon yang bercabang banyak, subpohon dapat digantikan daun bila tidak lagi menguntungkan. Ini adalah aturan operasional implementasi ini, bukan `ccp_alpha` dari scikit-learn.

In [13]:
"""Shared preprocessing and small from-scratch ID3, C4.5, and CART baselines."""

MISSING = "__MISSING__"
UNKNOWN = "__UNKNOWN__"


class SharedPreprocessor:
    """Median-impute numeric data and retain nominal categories as strings."""

    def fit(self, X):
        self.columns = list(X.columns)
        self.numeric = [c for c in self.columns if pd.api.types.is_numeric_dtype(X[c])]
        self.categorical = [c for c in self.columns if c not in self.numeric]
        self.numeric_fill = {
            c: X[c].median() if X[c].notna().any() else 0.0
            for c in self.numeric
        }
        self.categories = {
            c: set(X[c].dropna().astype(str).unique()) for c in self.categorical
        }
        return self

    def transform(self, X):
        result = X.reindex(columns=self.columns).copy()
        for col in self.numeric:
            result[col] = pd.to_numeric(result[col], errors="coerce").fillna(self.numeric_fill[col])
        for col in self.categorical:
            values = result[col].astype("string")
            values = values.fillna(MISSING)
            result[col] = values.where(values.isin(self.categories[col]) | values.eq(MISSING), UNKNOWN).astype(object)
        return result

    def fit_transform(self, X):
        return self.fit(X).transform(X)


def _entropy(zeros, ones):
    total = zeros + ones
    if total == 0:
        return 0.0
    p = zeros / total
    q = ones / total
    return -(p * np.log2(p) if p else 0.0) - (q * np.log2(q) if q else 0.0)


def _gini(zeros, ones):
    total = zeros + ones
    if total == 0:
        return 0.0
    p = zeros / total
    q = ones / total
    return 1.0 - p * p - q * q


@dataclass
class Node:
    prediction: int
    positive_probability: float
    n_samples: int
    weight_sum: float
    feature: str | None = None
    kind: str | None = None
    threshold: float | None = None
    categories: frozenset | None = None
    branches: dict | None = None


class BaselineTree:
    """Greedy binary classifier using information gain, gain ratio, or Gini."""

    def __init__(self, criterion="gini", max_depth=8, min_samples_leaf=40, min_samples_split=80):
        if criterion not in {"information_gain", "gain_ratio", "gini"}:
            raise ValueError("criterion must be information_gain, gain_ratio, or gini")
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split

    def fit(self, X, y, class_weight=None):
        self.columns = list(X.columns)
        self.numeric = [c for c in self.columns if pd.api.types.is_numeric_dtype(X[c])]
        self.values = {c: X[c].to_numpy() for c in self.columns}
        self.y = np.asarray(y)
        if set(np.unique(self.y)) - {0, 1}:
            raise ValueError("BaselineTree expects binary labels 0/1")
        self.y = self.y.astype(np.int8)
        class_weight = class_weight or {}
        if any(label not in {0, 1} or not np.isfinite(weight) or weight <= 0 for label, weight in class_weight.items()):
            raise ValueError("class_weight must map binary labels to finite positive weights")
        self.sample_weights = np.asarray([class_weight.get(label, 1.0) for label in self.y], dtype=float)
        self.total_weight = float(self.sample_weights.sum())
        self.root = self._grow(np.arange(len(self.y)), 0)
        return self

    def _weighted_counts(self, indices):
        labels = self.y[indices]
        weights = self.sample_weights[indices]
        return weights[labels == 0].sum(), weights[labels == 1].sum()

    def _impurity(self, zero, one):
        return _gini(zero, one) if self.criterion == "gini" else _entropy(zero, one)

    def _score(self, groups, parent_impurity, parent_weight):
        if len(groups) < 2 or any(len(group) < self.min_samples_leaf for group in groups):
            return None
        weighted = sum(
            self.sample_weights[group].sum() * self._impurity(*self._weighted_counts(group))
            for group in groups
        ) / parent_weight
        gain = parent_impurity - weighted
        if gain <= 1e-12:
            return None
        if self.criterion != "gain_ratio":
            return gain, gain
        split_info = _entropy(0, 0)
        for group in groups:
            p = self.sample_weights[group].sum() / parent_weight
            split_info -= p * np.log2(p)
        return (gain / split_info, gain) if split_info else None

    def _numeric_candidates(self, indices, col, parent_impurity, parent_weight):
        values = self.values[col][indices].astype(float)
        order = np.argsort(values, kind="stable")
        sorted_values = values[order]
        if len(sorted_values) < 2:
            return
        cuts = np.flatnonzero(sorted_values[:-1] < sorted_values[1:]) + 1
        if len(cuts) > 64:
            cuts = np.unique(cuts[np.linspace(0, len(cuts) - 1, 64).astype(int)])
        for cut in cuts:
            if cut < self.min_samples_leaf or len(indices) - cut < self.min_samples_leaf:
                continue
            threshold = (sorted_values[cut - 1] + sorted_values[cut]) / 2
            left = indices[order[:cut]]
            right = indices[order[cut:]]
            scores = self._score((left, right), parent_impurity, parent_weight)
            if scores is not None:
                score, gain = scores
                yield score, gain, col, "numeric", threshold, None, {"le": left, "gt": right}

    def _categorical_candidates(self, indices, col, parent_impurity, parent_weight):
        values = self.values[col][indices].astype(str)
        categories = np.unique(values)
        if len(categories) < 2:
            return
        if self.criterion != "gini":
            groups = {category: indices[values == category] for category in categories}
            scores = self._score(tuple(groups.values()), parent_impurity, parent_weight)
            if scores is not None:
                score, gain = scores
                yield score, gain, col, "multiway", None, None, groups
            return

        # CART tests unique binary partitions. Above 10 levels, use one-vs-rest
        # candidates to avoid exponential work on high-cardinality features.
        if len(categories) <= 10:
            # Canonical partitions always contain the first level on the left.
            subsets = (frozenset([categories[0], *combo]) for size in range(len(categories)) for combo in combinations(categories[1:], size))
        else:
            subsets = (frozenset([category]) for category in categories)
        seen = set()
        for left_categories in subsets:
            if len(left_categories) == len(categories):
                continue
            key = tuple(sorted(left_categories))
            if key in seen:
                continue
            seen.add(key)
            left_mask = np.isin(values, list(left_categories))
            left, right = indices[left_mask], indices[~left_mask]
            scores = self._score((left, right), parent_impurity, parent_weight)
            if scores is not None:
                score, gain = scores
                yield score, gain, col, "binary_category", None, left_categories, {"left": left, "right": right}

    def _grow(self, indices, depth):
        negatives, positives = self._weighted_counts(indices)
        majority = int(positives > negatives)
        weight_sum = negatives + positives
        node = Node(majority, positives / weight_sum, len(indices), weight_sum)
        if (self.max_depth is not None and depth >= self.max_depth) or len(indices) < self.min_samples_split or positives == 0 or negatives == 0:
            return node
        parent_impurity = self._impurity(negatives, positives)
        candidates = (
            candidate
            for col in self.columns
            for candidate in (
                self._numeric_candidates(indices, col, parent_impurity, weight_sum)
                if col in self.numeric
                else self._categorical_candidates(indices, col, parent_impurity, weight_sum)
            )
        )
        if self.criterion == "gain_ratio":
            candidates = list(candidates)
        if self.criterion == "gain_ratio" and candidates:
            best_gain_by_feature = {}
            for _, gain, col, *_ in candidates:
                best_gain_by_feature[col] = max(best_gain_by_feature.get(col, 0.0), gain)
            mean_gain = np.mean(list(best_gain_by_feature.values()))
            candidates = [candidate for candidate in candidates if candidate[1] >= mean_gain]
        best = max(candidates, key=lambda candidate: candidate[0], default=None)
        if best is None:
            return node
        _, _, node.feature, node.kind, node.threshold, node.categories, groups = best
        node.branches = {key: self._grow(group, depth + 1) for key, group in groups.items()}
        return node

    def _predict_one(self, row):
        node = self.root
        while node.branches is not None:
            value = row[node.feature]
            if node.kind == "numeric":
                key = "le" if float(value) <= node.threshold else "gt"
            elif node.kind == "binary_category":
                key = "left" if str(value) in node.categories else "right"
            else:
                key = str(value)
            child = node.branches.get(key)
            if child is None:
                return node.prediction
            node = child
        return node.prediction

    def predict(self, X):
        return [int(self._predict_one(row)) for row in X.to_dict(orient="records")]

    def _positive_probability_one(self, row):
        node = self.root
        while node.branches is not None:
            value = row[node.feature]
            if node.kind == "numeric":
                key = "le" if float(value) <= node.threshold else "gt"
            elif node.kind == "binary_category":
                key = "left" if str(value) in node.categories else "right"
            else:
                key = str(value)
            child = node.branches.get(key)
            if child is None:
                return node.positive_probability
            node = child
        return node.positive_probability

    def predict_proba(self, X):
        positive = np.asarray([
            self._positive_probability_one(row)
            for row in X.to_dict(orient="records")
        ])
        return np.column_stack((1.0 - positive, positive))

    def _prune_node(self, node, alpha, total_weight):
        leaf_error = node.weight_sum * min(node.positive_probability, 1.0 - node.positive_probability) / total_weight
        if node.branches is None:
            return leaf_error, 1

        subtree_error = 0.0
        leaves = 0
        for child in node.branches.values():
            child_error, child_leaves = self._prune_node(child, alpha, total_weight)
            subtree_error += child_error
            leaves += child_leaves

        if leaf_error + alpha <= subtree_error + alpha * leaves:
            node.feature = node.kind = node.threshold = node.categories = None
            node.branches = None
            return leaf_error, 1
        return subtree_error, leaves

    def pruned(self, alpha):
        if not np.isfinite(alpha) or alpha < 0:
            raise ValueError("alpha must be finite and non-negative")
        result = copy(self)
        result.root = deepcopy(self.root)
        self._prune_node(result.root, alpha, self.total_weight)
        return result

    def leaf_count(self):
        def count(node):
            return 1 if node.branches is None else sum(count(child) for child in node.branches.values())

        return count(self.root)


def fit_preprocessor(X_fit):
    return SharedPreprocessor().fit(X_fit)


def transform_features(preprocessor, X):
    return preprocessor.transform(X)


preprocessor = fit_preprocessor(X_train_raw)
X_train_model = transform_features(preprocessor, X_train_raw)
X_val_model = transform_features(preprocessor, X_val_raw)
assert len(X_train_model) == len(y_train) and len(X_val_model) == len(y_val)

Preprocessor dilatih pada bagian latih holdout. Dalam validasi silang, setiap fold membuat preprocessor baru dari bagian latih fold sehingga median dan daftar kategori tidak bocor dari fold validasi.

---

# Modeling and Validation <a name="7"></a>

---

Baseline ID3, C4.5, dan CART memakai `max_depth=8`, `min_samples_split=80`, serta `min_samples_leaf=40`. Untuk DTL-v08, lima fold stratifikasi pada bagian latih membandingkan tiga konfigurasi pertumbuhan, sebelas nilai `alpha`, dan ambang probabilitas 0,20–0,80 dengan langkah 0,02. Prediksi *out-of-fold* dinilai menggunakan Macro F1. Bila seri, dipilih jumlah daun rerata lebih sedikit, lalu `alpha` dan ambang lebih besar. Holdout tidak ikut memilih kandidat.

In [14]:
tree_configs = {
    'ID3': {'criterion': 'information_gain'},
    'C4.5': {'criterion': 'gain_ratio'},
    'CART': {'criterion': 'gini'},
}
CART_TREE_CONFIGS = ((None, 2, 1), (12, 40, 20), (8, 80, 40))

def threshold_predictions(probabilities, threshold):
    return (np.asarray(probabilities) >= threshold).astype(np.int8).tolist()


def prepare_fold(X_train, X_valid):
    preprocessor = SharedPreprocessor().fit(X_train)
    return preprocessor.transform(X_train), preprocessor.transform(X_valid)


def tune_cart_pruning(X, y, n_splits=5, tree_configs=CART_TREE_CONFIGS):
    y = np.asarray(y, dtype=np.int8)
    alphas = (0.0, 0.0001, 0.0005, 0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1, 0.25)
    thresholds = np.round(np.arange(0.20, 0.8001, 0.02), 2)
    candidates = [
        (max_depth, min_samples_split, min_samples_leaf, alpha)
        for max_depth, min_samples_split, min_samples_leaf in tree_configs
        for alpha in alphas
    ]
    probabilities = {candidate: np.empty(len(y), dtype=float) for candidate in candidates}
    leaves = {candidate: [] for candidate in candidates}
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for train_idx, valid_idx in splitter.split(X, y):
        X_train, X_valid = prepare_fold(X.iloc[train_idx], X.iloc[valid_idx])
        for max_depth, min_samples_split, min_samples_leaf in tree_configs:
            model = BaselineTree(
                criterion="gini", max_depth=max_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=min_samples_leaf,
            ).fit(X_train, y[train_idx])
            for alpha in alphas:
                candidate = (max_depth, min_samples_split, min_samples_leaf, alpha)
                tree = model.pruned(alpha)
                probabilities[candidate][valid_idx] = tree.predict_proba(X_valid)[:, 1]
                leaves[candidate].append(tree.leaf_count())

    results = []
    for max_depth, min_samples_split, min_samples_leaf, alpha in candidates:
        for threshold in thresholds:
            candidate = (max_depth, min_samples_split, min_samples_leaf, alpha)
            prediction = threshold_predictions(probabilities[candidate], threshold)
            results.append({
                "max_depth": max_depth,
                "min_samples_split": min_samples_split,
                "min_samples_leaf": min_samples_leaf,
                "alpha": alpha,
                "threshold": float(threshold),
                "macro_f1": f1_score(y, prediction, labels=[0, 1], average="macro", zero_division=0),
                "mean_leaves": float(np.mean(leaves[candidate])),
            })
    best = max(results, key=lambda result: (result["macro_f1"], -result["mean_leaves"], result["alpha"], result["threshold"]))
    return best, pd.DataFrame(results)



def train_model(X, y, config):
    model = BaselineTree(
        criterion=config['criterion'],
        max_depth=config.get('max_depth', 8),
        min_samples_split=config.get('min_samples_split', 80),
        min_samples_leaf=config.get('min_samples_leaf', 40),
    ).fit(X, y)
    if 'alpha' in config:
        model = model.pruned(config['alpha'])
    return (model, config['threshold']) if 'threshold' in config else model


def predict_model(model, X):
    if isinstance(model, tuple):
        tree, threshold = model
        return threshold_predictions(tree.predict_proba(X)[:, 1], threshold)
    return model.predict(X)

Pemilihan kandidat dilakukan melalui Macro F1 validasi silang. Skor holdout dilaporkan setelah parameter tetap. Nilai yang terpilih dapat berbeda dari hasil historis bila dataset atau lingkungan berubah.

In [15]:
tree_results = []
tree_models = {}
for variant, config in tree_configs.items():
    candidate = train_model(X_train_model, y_train, config)
    predictions = np.asarray(predict_model(candidate, X_val_model))
    if predictions.shape != y_val.shape or not np.isin(predictions, [0, 1]).all():
        raise ValueError('Setiap baseline harus menghasilkan satu label biner per baris.')
    tree_models[variant] = candidate
    tree_results.append({
        'variant': variant,
        'macro_f1': f1_score(y_val, predictions, labels=[0, 1],
                             average='macro', zero_division=0),
    })
tree_scores = pd.DataFrame(tree_results).set_index('variant')
display(tree_scores.sort_values('macro_f1', ascending=False))

best, pruning_results = tune_cart_pruning(X_train_raw, y_train)
selected_variant = 'CART'
selected_config = {
    'criterion': 'gini',
    'max_depth': best['max_depth'],
    'min_samples_split': best['min_samples_split'],
    'min_samples_leaf': best['min_samples_leaf'],
    'alpha': best['alpha'],
    'threshold': best['threshold'],
}
print('DTL-v08 — parameter terpilih dari 5-fold CV:', selected_config)
print('Macro F1 out-of-fold:', round(best['macro_f1'], 4))
display(pruning_results.sort_values('macro_f1', ascending=False).head(10))

model = train_model(X_train_model, y_train, selected_config)
y_val_pred = np.asarray(predict_model(model, X_val_model), dtype=int)
validation_macro_f1 = f1_score(y_val, y_val_pred, labels=[0, 1],
                                average='macro', zero_division=0)
print(f'Macro F1 holdout DTL-v08: {validation_macro_f1:.4f}')
precision, recall, class_f1, support = precision_recall_fscore_support(
    y_val, y_val_pred, labels=[0, 1], zero_division=0
)
display(pd.DataFrame({'precision': precision, 'recall': recall,
                      'f1': class_f1, 'support': support}, index=[0, 1]))
display(pd.DataFrame(confusion_matrix(y_val, y_val_pred, labels=[0, 1]),
                     index=['aktual 0', 'aktual 1'],
                     columns=['prediksi 0', 'prediksi 1']))

,macro_f1
variant,
CART,0.812727
ID3,0.808621
C4.5,0.706141


DTL-v08 — parameter terpilih dari 5-fold CV: {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'alpha': 0.0001, 'threshold': 0.62}
Macro F1 out-of-fold: 0.832


,max_depth,min_samples_split,min_samples_leaf,alpha,threshold,macro_f1,mean_leaves
52,NaN,2,1,0.0001,0.62,0.832007,371.2
47,NaN,2,1,0.0001,0.52,0.831927,371.2
46,NaN,2,1,0.0001,0.50,0.831927,371.2
43,NaN,2,1,0.0001,0.44,0.831927,371.2
48,NaN,2,1,0.0001,0.54,0.831927,371.2
49,NaN,2,1,0.0001,0.56,0.831927,371.2
50,NaN,2,1,0.0001,0.58,0.831927,371.2
51,NaN,2,1,0.0001,0.60,0.831927,371.2
45,NaN,2,1,0.0001,0.48,0.831927,371.2
44,NaN,2,1,0.0001,0.46,0.831927,371.2


Macro F1 holdout DTL-v08: 0.8371


,precision,recall,f1,support
0,0.874583,0.926379,0.899736,3681
1,0.828046,0.727425,0.774481,1794


,prediksi 0,prediksi 1
aktual 0,3410,271
aktual 1,489,1305


Tabel baseline menunjukkan perbedaan ID3, C4.5, dan CART pada holdout yang sama. Tabel kandidat menampilkan hasil *out-of-fold* untuk konfigurasi CART. Skor holdout DTL-v08 dihitung setelah `alpha` dan ambang ditetapkan. Nilai ini adalah hasil eksekusi, bukan klaim bahwa pemangkasan selalu lebih baik daripada baseline.

---

# Comparison with scikit-learn <a name="8"></a>

---

Pembanding memakai `DecisionTreeClassifier` pada baris latih dan validasi yang sama. Angka diimputasi dengan median dan kategori di-*one-hot encode* dari bagian latih. `entropy` hanya pendekatan untuk ID3 dan C4.5 karena scikit-learn tidak memiliki pemisahan multi-cabang ID3 atau *gain ratio* C4.5. Pembanding ini bukan DTL-v08 dan tidak dipakai untuk memilih pipeline final.

In [16]:
comparison_preprocessor = fit_preprocessor(X_train_raw)
X_train_prepared = transform_features(comparison_preprocessor, X_train_raw)
X_val_prepared = transform_features(comparison_preprocessor, X_val_raw)
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
categorical = comparison_preprocessor.categorical
numeric = comparison_preprocessor.numeric
if categorical:
    encoded_train = encoder.fit_transform(X_train_prepared[categorical])
    encoded_val = encoder.transform(X_val_prepared[categorical])
else:
    encoded_train = np.empty((len(X_train_prepared), 0))
    encoded_val = np.empty((len(X_val_prepared), 0))
X_train_comparison = np.column_stack((X_train_prepared[numeric].to_numpy(), encoded_train))
X_val_comparison = np.column_stack((X_val_prepared[numeric].to_numpy(), encoded_val))

comparison_results = []
for variant, criterion, alpha in (('ID3', 'entropy', 0.0),
                                  ('C4.5', 'entropy', 0.001),
                                  ('CART', 'gini', 0.0)):
    comparator = DecisionTreeClassifier(
        criterion=criterion, ccp_alpha=alpha, max_depth=8,
        min_samples_split=80, min_samples_leaf=40, random_state=CFG.SEED
    ).fit(X_train_comparison, y_train)
    predictions = comparator.predict(X_val_comparison)
    comparison_results.append({
        'variant': variant,
        'macro_f1': f1_score(y_val, predictions, labels=[0, 1],
                             average='macro', zero_division=0),
    })
display(pd.DataFrame(comparison_results).set_index('variant'))

,macro_f1
variant,
ID3,0.812807
C4.5,0.811554
CART,0.804924


Perbedaan skor tidak semata-mata mencerminkan kualitas implementasi karena cara menangani kategori, bentuk cabang, dan kriteria pemisahan tidak sepenuhnya identik.

---

# Saving and Loading Models <a name="9"></a>

---

Setelah evaluasi holdout, imputasi dilatih ulang pada seluruh `train.csv`, disusul CART dengan parameter CV terpilih. Artefak menyimpan preprocessor dan model bersama agar kategori baru serta median numerik diperlakukan konsisten. `pickle` hanya aman untuk artefak buatan sendiri atau sumber tepercaya. Jalankan sel definisi kelas sebelum memuatnya pada sesi Python baru.

In [17]:
X_full_raw = train_df[feature_columns].copy()
y_full = train_df['is_canceled'].to_numpy(dtype=int)
final_preprocessor = fit_preprocessor(X_full_raw)
X_full_model = transform_features(final_preprocessor, X_full_raw)
final_model = train_model(X_full_model, y_full, selected_config)

CFG.MODEL_DIR.mkdir(parents=True, exist_ok=True)

def save_model_bundle(path, preprocessor, model):
    with (path / 'bundle.pkl').open('wb') as file:
        pickle.dump((preprocessor, model), file)


def load_model_bundle(path):
    with (path / 'bundle.pkl').open('rb') as file:
        return pickle.load(file)


save_model_bundle(CFG.MODEL_DIR, final_preprocessor, final_model)
loaded_preprocessor, loaded_model = load_model_bundle(CFG.MODEL_DIR)
assert np.array_equal(
    predict_model(final_model, transform_features(final_preprocessor, X_val_raw)),
    predict_model(loaded_model, transform_features(loaded_preprocessor, X_val_raw)),
)
print('Artefak tersimpan:', CFG.MODEL_DIR / 'bundle.pkl')

Artefak tersimpan: c:\Users\aufar\Institut Teknologi Bandung\IF3170_G03_FiveAgents\output\final\model\dtl\bundle.pkl


## Bonus DTL (Gambar Percabangan Tree)

`CFG.TREE_TOP_N` mengatur jumlah split pertama yang ditampilkan, dihitung breadth-first dari root pada CART final yang sudah di-prune. Leaf tetap ditampilkan, sedangkan node yang belum dibuka karena batas N diberi tanda `cut`. Aturan cabang ditulis di dalam node anak agar tidak bertumpuk dengan garis. Himpunan kategori panjang diringkas, tetapi aturan lengkap dapat dilihat dengan mengarahkan penunjuk ke label pada SVG. Berkas SVG tersimpan di folder model dan ditampilkan di bawah.

In [ ]:
def render_tree_svg(root, top_n, path):
    if isinstance(top_n, bool) or not isinstance(top_n, int) or top_n < 1:
        raise ValueError('top_n must be a positive integer.')
    queue, shown = [root], set()
    for node in queue:
        if node.branches is None or len(shown) >= top_n:
            continue
        shown.add(id(node))
        queue.extend(node.branches.values())

    nodes, edges = [], []
    leaf_x = 0
    def place(node, depth, incoming=('root', 'root')):
        nonlocal leaf_x
        if id(node) in shown:
            children = []
            for branch, child in node.branches.items():
                if node.kind == 'numeric':
                    label = f'≤ {node.threshold:.4g}' if branch == 'le' else f'> {node.threshold:.4g}'
                    summary = label
                elif node.kind == 'binary_category':
                    categories = sorted(node.categories)
                    prefix = 'in ' if branch == 'left' else 'not in '
                    label = prefix + '{' + ', '.join(categories) + '}'
                    summary = prefix + '{' + categories[0] + (f', +{len(categories) - 1}' if len(categories) > 1 else '') + '}'
                else:
                    label = str(branch)
                    summary = label
                children.append(place(child, depth + 1, (summary, label)))
            x = (children[0] + children[-1]) / 2
            for child_x in children:
                edges.append((x, depth, child_x))
            title = str(node.feature)
        else:
            x = leaf_x * 240 + 120
            leaf_x += 1
            title = 'cut (more splits)' if node.branches is not None else 'leaf'
        nodes.append((x, depth, title, node, incoming))
        return x

    place(root, 0)
    height = (max(depth for _, depth, *_ in nodes) + 1) * 160
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{leaf_x * 240}" height="{height}" viewBox="0 0 {leaf_x * 240} {height}">',
             '<rect width="100%" height="100%" fill="white"/>']
    for x, depth, child_x in edges:
        y = depth * 160 + 100
        parts.append(f'<path d="M {x} {y} L {child_x} {y + 90}" stroke="#64748b" fill="none"/>')
    for x, depth, title, node, incoming in nodes:
        y = depth * 160 + 30
        color = '#dbeafe' if id(node) in shown else '#f1f5f9'
        parts.append(f'<rect x="{x - 105}" y="{y}" width="210" height="70" rx="7" fill="{color}" stroke="#64748b"/>')
        rule, full_rule = incoming
        rule = rule if len(rule) <= 26 else rule[:25] + '…'
        parts.append(f'<text x="{x}" y="{y + 18}" text-anchor="middle" font-size="11" fill="#334155">{escape(rule)}<title>{escape(full_rule)}</title></text>')
        parts.append(f'<text x="{x}" y="{y + 40}" text-anchor="middle" font-size="12" font-weight="bold">{escape(title[:26])}</text>')
        parts.append(f'<text x="{x}" y="{y + 59}" text-anchor="middle" font-size="11">n={node.n_samples} | p1={node.positive_probability:.3f}</text>')
    parts.append('</svg>')
    path.write_text('\n'.join(parts), encoding='utf-8')

tree_image = CFG.MODEL_DIR / f'tree_top_{CFG.TREE_TOP_N}.svg'
render_tree_svg(final_model[0].root, CFG.TREE_TOP_N, tree_image)
print('Gambar percabangan:', tree_image)
from IPython.display import SVG
display(SVG(filename=str(tree_image)))

Uji round-trip di atas memastikan prediksi model sebelum dan sesudah pemuatan sama pada sampel validasi. Karena model akhir sudah dilatih ulang pada seluruh data, sampel ini hanya menguji serialisasi, bukan menghitung ulang skor holdout.

---

# Kaggle Submission <a name="10"></a>

---

Prediksi dibuat dari artefak yang baru dimuat. ID dan urutan test dipertahankan. Label validasi maupun kolom target test tidak digunakan saat pelatihan akhir.

In [19]:
X_test_final = transform_features(loaded_preprocessor, X_test_raw)
test_predictions = np.asarray(predict_model(loaded_model, X_test_final))
if test_predictions.ndim != 1 or len(test_predictions) != len(test_df) or not np.isin(test_predictions, [0, 1]).all():
    raise ValueError('Test prediction must contain one label 0 or 1 per test row.')
test_predictions = test_predictions.astype(int)

submission = pd.DataFrame({'id': test_df['id'].to_numpy(), 'is_canceled': test_predictions})
if submission.columns.tolist() != ['id', 'is_canceled'] or not submission['id'].equals(test_df['id'].reset_index(drop=True)):
    raise ValueError('Submission columns, IDs, and row order must match test.csv exactly.')
submission_path = CFG.OUTPUT_DIR / f'submission_{CFG.MODEL_NAME}.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} rows to {submission_path}')

Saved 6844 rows to c:\Users\aufar\Institut Teknologi Bandung\IF3170_G03_FiveAgents\output\submission_dtl.csv


Berkas `output/submission_dtl.csv` memuat hanya `id,is_canceled`, tepat satu prediksi 0 atau 1 per baris test dalam urutan semula.

---

# Error Analysis <a name="11"></a>

---

Analisis berikut memakai prediksi holdout **sebelum** refit pada seluruh data. *False positive* berarti pemesanan yang tidak batal diprediksi batal. *False negative* berarti pembatalan terlewat.

In [20]:
error_analysis = X_val_raw.copy()
error_analysis['actual'] = y_val
error_analysis['predicted'] = y_val_pred
error_analysis['error_type'] = np.select(
    [error_analysis['actual'].eq(0) & error_analysis['predicted'].eq(1),
     error_analysis['actual'].eq(1) & error_analysis['predicted'].eq(0)],
    ['false_positive', 'false_negative'],
    default='correct',
)
display(error_analysis.loc[error_analysis['error_type'] != 'correct'].head(20))
display(error_analysis['error_type'].value_counts())

,num_adults,num_children,weekend_nights,weekday_nights,meal_package,needs_parking,room_category,days_before_arrival,arrival_year,arrival_month,arrival_day,booking_channel,is_returning_guest,prior_cancellations,prior_completed_bookings,nightly_rate_idr,special_requests,actual,predicted,error_type
69,2.0,0.0,2,4,Breakfast,0.0,Type G,26.0,2024,3,11,Online Travel Agent,0,0,0,1697000.0,0.0,0,1,false_positive
91,2.0,0.0,0,3,Breakfast,0.0,NaN,24.0,2024,10,3,Online Travel Agent,0,0,0,NaN,0.0,0,1,false_positive
136,2.0,0.0,0,1,Breakfast,0.0,Type G,88.0,2024,9,15,Online Travel Agent,0,0,0,2395000.0,1.0,1,0,false_negative
139,2.0,0.0,0,3,Breakfast,0.0,Type G,125.0,2024,9,28,Online Travel Agent,0,0,0,2301000.0,2.0,1,0,false_negative
149,2.0,0.0,1,2,Breakfast,0.0,Type B,98.0,2024,4,3,Offline Agent,0,0,0,NaN,0.0,1,0,false_negative
156,NaN,0.0,1,2,Breakfast,0.0,NaN,155.0,2024,8,14,Offline Agent,0,0,0,NaN,0.0,1,0,false_negative
161,2.0,0.0,0,3,Breakfast,0.0,Type G,80.0,2024,9,7,Online Travel Agent,0,0,0,2404000.0,1.0,1,0,false_negative
201,2.0,0.0,0,3,NaN,NaN,Type G,6.0,2024,3,22,Online Travel Agent,0,0,0,2492000.0,0.0,0,1,false_positive
301,2.0,0.0,2,5,Breakfast,0.0,Type G,72.0,2024,9,3,Online Travel Agent,0,0,0,2413000.0,1.0,1,0,false_negative
310,2.0,0.0,1,4,Breakfast,0.0,Type B,NaN,2024,10,4,Offline Agent,0,0,0,1490000.0,0.0,1,0,false_negative


error_type
correct           4715
false_negative     489
false_positive     271
Name: count, dtype: int64

Jumlah dan contoh kesalahan membantu menyusun hipotesis eksperimen berikutnya. Jangan memakai pola holdout untuk menyetel ulang DTL-v08 lalu melaporkan skor pada holdout yang sama sebagai evaluasi independen.